# 00 — Tensor basics

Everything you need to write `hedging_gains` in `dhbench/pnl.py`. Nothing more.

**How this works:** learn one small thing, then immediately practise that one thing.
Nine rounds. Each practice cell has a `TODO` and the answer to expect, so you can check
yourself without asking me.

Run cells with `Shift+Enter`. If VSCode asks for a kernel, pick the one in `.venv`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # hush TensorFlow's startup noise

import tensorflow as tf
tf.__version__

---
# Round 1 — Shape

## Learn

A tensor is a grid of numbers. Put it on the last line of a cell and Jupyter shows you
the values, the **shape**, and the dtype — all at once.

In [ ]:
x = tf.constant([[1., 2., 3., 4.],
                 [5., 6., 7., 8.]])
x

`shape=(2, 4)` means **2 rows, 4 columns**. The two directions are numbered:

- **axis 0** = down the rows
- **axis 1** = across the columns

In this project: **rows are paths, columns are time.**

*(Watch the shape in every output below. "It didn't crash" is not evidence it is right.)*

## Practice 1

Show `y`. Say its shape out loud *before* you run it.

In [ ]:
y = tf.constant([[1., 2., 3., 4., 5.],
                 [6., 7., 8., 9., 10.]])

# TODO

# expect shape=(2, 5)

---
# Round 2 — Drop the first column

## Learn

Slicing is `x[rows, columns]`.

- `:` means **all of them**
- `1:` means **from index 1 onwards** — drop the first

In [ ]:
x[:, 1:]

## Practice 2

Same thing to `y`.

In [ ]:
# TODO: y without its first column

# expect [[2. 3. 4. 5.], [7. 8. 9. 10.]]  shape=(2, 4)

---
# Round 3 — Drop the last column

## Learn

`:-1` means **up to but not including the last** — drop the last.

In [ ]:
x[:, :-1]

## Practice 3

Same thing to `y`.

In [ ]:
# TODO: y without its last column

# expect [[1. 2. 3. 4.], [6. 7. 8. 9.]]  shape=(2, 4)

---
# Round 4 — Subtract the two slices

## Learn

Both slices have the **same shape**, and they are the same list shifted by one:

```
drop last   1   2   3        <- the "before" value
drop first      2   3   4    <- the "after" value
```

Subtract, and you get the change from each value to the next. With prices, that is the
price move.

In [ ]:
prices = tf.constant([[100., 110., 105., 120.]])

prices[:, 1:] - prices[:, :-1]

**4 prices give 3 moves.** 100→110 is +10, 110→105 is −5, 105→120 is +15.

This is exactly why `spot` has `n_steps + 1` columns but `delta` has only `n_steps`:
one decision per *gap*, not per price.

## Practice 4

In [ ]:
p = tf.constant([[50., 55., 53.]])

# TODO: the moves of p

# expect [[5. -2.]]  shape=(1, 2)

---
# Round 5 — Adding up, and `axis`

## Learn

`tf.reduce_sum` adds numbers up. `axis` says **which direction**.

> **The rule: the axis you name is the axis that disappears.**

In [ ]:
tf.reduce_sum(x, axis=0)   # rows disappear -> add DOWN each column

In [ ]:
tf.reduce_sum(x, axis=1)   # columns disappear -> add ACROSS each row

Started from `(2, 4)`:

| call | axis named | length of that axis | result shape |
|:--|:--|:--|:--|
| `axis=0` | 0 (rows) | 2 | `(4,)` |
| `axis=1` | 1 (columns) | 4 | `(2,)` |

**For us:** rows are paths, columns are time. We want *one number per path*, shape
`(n_paths,)`. So the **time** axis is the one that must disappear.

## Practice 5

`z` is `(2, 3)`. Predict both before running.

In [ ]:
z = tf.constant([[1., 2., 3.],
                 [4., 5., 6.]])

# TODO: sum z so the ROWS disappear      -> expect [5. 7. 9.]  shape=(3,)


# TODO: sum z so the COLUMNS disappear   -> expect [6. 15.]    shape=(2,)


---
# Round 6 — `axis=-1`

## Learn

`-1` means **the last axis**, counting backwards. For a 2-D grid that is the same as
`axis=1`.

In [ ]:
tf.reduce_sum(x, axis=-1)

**Use `axis=-1` as your habit.** Later an extra dimension may appear at the front; `-1`
keeps meaning "time", while `1` would silently start meaning something else.

## Practice 6

Redo the second half of Practice 5 with `axis=-1`.

In [ ]:
# TODO

# expect [6. 15.]  shape=(2,)

---
# Round 7 — Multiplying two grids

## Learn

`*` between two grids of the **same shape** multiplies position by position. It is *not*
matrix multiplication.

In [ ]:
tf.constant([[1., 2., 3.]]) * tf.constant([[10., 20., 30.]])

**A trap worth seeing now.** If the shapes *don't* match, TensorFlow often does **not**
error — it stretches one side to fit and hands you a plausible wrong answer. Hence:
watch the shape on every output.

In [ ]:
tf.constant([[1., 2., 3.]]) * tf.constant([2.])   # no error!

## Practice 7

Multiply the position held by the price move, gap by gap.

In [ ]:
held = tf.constant([[1., 2.]])     # position held over each gap
gaps = tf.constant([[5., -2.]])    # price move over each gap

# TODO

# expect [[5. -4.]]  shape=(1, 2)

---
# Round 8 — Put it together: `hedging_gains`

## Learn

$$\text{gains} = \sum_{i=0}^{n-1} \delta_i \,(S_{i+1} - S_i)$$

In words: **for each gap, multiply the position you were holding by the price move over
that gap; add them up, separately for each path.**

That is Round 4 (moves) → Round 7 (multiply) → Round 6 (sum over time).

## Practice 8 — one path

Check by hand first:

| gap | held | move | gain |
|:--|:--|:--|:--|
| 100 → 110 | 1 | +10 | +10 |
| 110 → 105 | 2 | −5 | −10 |
| 105 → 120 | −1 | +15 | −15 |

**Total = −15.**

In [ ]:
spot  = tf.constant([[100., 110., 105., 120.]])   # 1 path, 4 prices
delta = tf.constant([[1.,   2.,  -1.]])           # 1 path, 3 positions

# TODO: moves -> multiply -> sum over time

# expect [-15.]  shape=(1,)

## Practice 9 — two paths

One path hides an `axis` mistake, because `(1,)` and `(3,)` both let you index `[0]`.
Run the **same code** on two paths.

If you get `shape=(3,)` instead of `(2,)`, you summed the wrong axis — back to Round 5.

In [ ]:
spot2  = tf.constant([[100., 110., 105., 120.],
                      [100.,  90.,  95.,  85.]])
delta2 = tf.constant([[1., 2., -1.],
                      [0., 1.,  1.]])

# TODO: same code, on spot2 and delta2

# expect [-15. -5.]  shape=(2,)

---
# Round 9 — Move it into the real file

Once Practice 9 gives `[-15. -5.]`, open `dhbench/pnl.py`, find `hedging_gains`, and
replace `raise NotImplementedError` with the code you wrote — using the argument names
`spot` and `delta`.

**No `.numpy()` in there.** In the notebook it is just a display convenience; inside
`dhbench/` it severs the gradient tape and Stage 2 silently stops training.

Then run the three tests that don't need the (not yet written) GBM simulator:

```
python -m pytest tests/test_rung2_pnl_accounting.py -k hedging_gains -v
```

Three passes. Then tell me — `transaction_costs` is next, and that one has the
interesting bug in it.